In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.optim import Adam
import random
from collections import deque, namedtuple
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ====================== 1. 配置参数 ======================
class Config:
    # 轨迹参数
    traj_length = 50
    traj_dim = 3
    obstacle_num = 5
    obstacle_dim = 10 #同模型
    
    goal_pos = np.array([10.0, 10.0, 0.0])
    obstacle_pos = np.array([5.0, 5.0, 0.0])
    obstacle_radius = 1.5
    start_pos = np.array([0.0, 0.0, 0.0])
    
    # Diffusion 参数
    denoise_steps = 10
    guidance_scale = 1.0
    beta_start = 0.0001
    beta_end = 0.02
    betas = np.linspace(beta_start, beta_end, denoise_steps)
    alphas = 1.0 - betas
    alphas_cumprod = np.cumprod(alphas)
    alphas_cumprod_prev = np.concatenate([np.array([1.0]), alphas_cumprod[:-1]])
    
    # SAC 参数
    # state_dim = 1 + 1 + traj_length*traj_dim + traj_length*traj_dim + traj_dim + 1  # 额外加一个成功标志
    state_dim = traj_length*traj_dim + obstacle_num*obstacle_dim + 1 + 1                 #steps,sigma_t
    action_dim = 6  # 3维方向 + 3维权重
    
    # 网络结构
    hidden_dim = 256
    learning_rate = 3e-4
    
    # SAC 超参数
    gamma = 0.99  # 折扣因子
    tau = 0.005  # 目标网络软更新系数
    alpha = 0.2  # 温度参数初始值（熵系数）
    learn_alpha = True  # 是否自动学习alpha
    target_entropy = -action_dim  # 目标熵
    
    # 训练参数
    buffer_size = 100000
    batch_size = 256
    warmup_steps = 5000  # 增加预热步数
    max_steps = 100000  # 最大训练步数
    update_every = 50  # 每N步更新一次
    num_updates = 1  # 每次更新迭代次数
    
    # 环境参数
    max_episode_length = denoise_steps
    success_threshold = 0.5  # 距离小于此值算成功

# ====================== 2. Diffusion 基础模型 ======================
class DiffusionModel(nn.Module):
    """改进的扩散噪声预测模型"""
    def __init__(self, traj_length=50, traj_dim=3, denoise_steps=10):
        super().__init__()
        self.traj_length = traj_length
        self.traj_dim = traj_dim
        self.denoise_steps = denoise_steps
        
        # 时间步嵌入
        self.time_emb = nn.Sequential(
            nn.Embedding(denoise_steps+1, 128),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 128)
        )
        
        # 轨迹编码器
        self.traj_encoder = nn.Sequential(
            nn.Linear(traj_length*traj_dim, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, 128)
        )
        
        # 噪声预测网络
        self.noise_predictor = nn.Sequential(
            nn.Linear(128 + 128, 256),  # 轨迹特征 + 时间特征
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Linear(256, traj_length*traj_dim)
        )
        
        # 初始化权重
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
    
    def forward(self, x_t, t):
        # 轨迹编码
        traj_feat = self.traj_encoder(x_t)
        
        # 时间编码
        t_emb = self.time_emb(t)
        
        # 合并特征并预测噪声
        combined = torch.cat([traj_feat, t_emb], dim=-1)
        noise_pred = self.noise_predictor(combined)
        
        return noise_pred
    
    @torch.no_grad()
    def predict_noise(self, x_t, step, device):
        """预测噪声（推理模式）"""
        self.eval()
        x_t_flat = torch.tensor(x_t.flatten(), dtype=torch.float32).unsqueeze(0).to(device)
        t = torch.tensor([step], dtype=torch.long).to(device)
        noise_pred = self.forward(x_t_flat, t)
        return noise_pred.cpu().numpy().reshape(self.traj_length, self.traj_dim)
    
    def denoise_step(self, x_t, noise_guided, step, add_noise=True):
        """单步去噪"""
        t = step - 1
        alpha = Config.alphas[t]
        alpha_cumprod = Config.alphas_cumprod[t]
        alpha_cumprod_prev = Config.alphas_cumprod_prev[t]
        
        x_t_flat = x_t.flatten()
        noise_guided_flat = noise_guided.flatten()
        
        # 去噪公式
        x_prev = (1/np.sqrt(alpha)) * (
            x_t_flat - (1-alpha)/np.sqrt(1-alpha_cumprod) * noise_guided_flat
        )
        
        # 训练时加噪声，推理时不加
        if add_noise and step > 1:
            noise = np.random.randn(*x_prev.shape) * 0.1
            x_prev += np.sqrt(1 - alpha_cumprod_prev) * noise
        
        return x_prev.reshape(self.traj_length, self.traj_dim)

# ====================== 3. SAC 模型组件 ======================
class GaussianPolicy(nn.Module):
    """SAC策略网络（高斯策略）"""
    def __init__(self, state_dim, action_dim, hidden_dim=256, log_std_min=-20, log_std_max=2):
        super().__init__()
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max
        
        # 共享特征提取层
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
        )
        
        # 均值网络（输出经过约束的动作）
        self.mean_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.SiLU(),
            nn.Linear(hidden_dim//2, action_dim)
        )
        
        # 对数标准差网络
        self.log_std_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.SiLU(),
            nn.Linear(hidden_dim//2, action_dim)
        )
        
        # 初始化
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=0.01)
            nn.init.constant_(module.bias, 0)
    
    def forward(self, state, deterministic=False, with_logprob=True):
        """前向传播"""
        # state_dim = state_dim + max_step(10)
        # state = torch.cat([state, max_step],dim=-1)
        features = self.shared(state)
        
        # 计算均值（带约束）
        mean = self.mean_net(features)
        mean_dir = mean[:, :3] / (torch.norm(mean[:, :3], dim=1, keepdim=True) + 1e-8)
        mean_weight = torch.sigmoid(mean[:, 3:])  # 权重在[0,1]之间
        mean = torch.cat([mean_dir, mean_weight], dim=1)
        
        # 计算对数标准差（带约束）
        log_std = self.log_std_net(features)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        std = torch.exp(log_std)
        
        # 确定性输出（用于评估）
        if deterministic:
            return mean, None, None
        
        # 随机采样（使用重参数化技巧）
        normal = torch.distributions.Normal(mean, std)
        z = normal.rsample()  # 重参数化采样
        
        # 对采样结果进行约束
        action_dir = z[:, :3] / (torch.norm(z[:, :3], dim=1, keepdim=True) + 1e-8)
        action_weight = torch.sigmoid(z[:, 3:])  # 确保权重在[0,1]
        action = torch.cat([action_dir, action_weight], dim=1)
        
        # 计算对数概率
        if with_logprob:
            # 计算变换后的对数概率（考虑sigmoid的雅可比行列式）
            log_prob = normal.log_prob(z)
            
            # 方向部分的对数概率修正（球面坐标）
            dir_log_prob = log_prob[:, :3].sum(dim=1, keepdim=True)
            
            # 权重部分的对数概率修正（sigmoid变换）
            weight_log_prob = log_prob[:, 3:] - torch.log(action_weight * (1 - action_weight) + 1e-8)
            weight_log_prob = weight_log_prob.sum(dim=1, keepdim=True)
            
            log_prob = dir_log_prob + weight_log_prob
            
            return action, log_prob, z
        else:
            return action

class QNetwork(nn.Module):
    """SAC Q值网络"""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        # nn.Linear(state_dim + action_dim + max_step, hidden_dim),
        
        self.q_net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0)
    
    def forward(self, state, action):
        # x = torch.cat([state, action, max_step], dim=-1)
        x = torch.cat([state, action], dim=-1)
        return self.q_net(x)

# ====================== 4. 环境与工具函数 ======================
def get_sigma(step):
    """获取噪声强度σ"""
    t = step - 1
    return np.sqrt(1 - Config.alphas_cumprod[t])

def decode_action_to_cond_grad(action, step):
    """将6维动作解码为高维梯度"""
    sigma_t = get_sigma(step)
    dir_vec = action[:3]
    weight_vec = action[3:]
    
    # 轨迹分段（三段式）
    seg_len = Config.traj_length // 3
    seg1_end = seg_len
    seg2_end = 2 * seg_len
    
    # 生成分段权重矩阵
    weight_mat = np.zeros((Config.traj_length, Config.traj_dim))
    weight_mat[:seg1_end, :] = weight_vec[0]
    weight_mat[seg1_end:seg2_end, :] = weight_vec[1]
    weight_mat[seg2_end:, :] = weight_vec[2]
    
    # 生成条件梯度
    cond_grad = dir_vec * weight_mat * sigma_t
    
    return cond_grad

def compute_traj_curvature(traj):
    """计算轨迹曲率"""
    if len(traj) < 3:
        return 0.0
    
    # 计算切线向量
    tangents = np.diff(traj, axis=0)
    tangent_norms = np.linalg.norm(tangents, axis=1)
    
    # 避免除零
    tangent_norms[tangent_norms < 1e-8] = 1e-8
    unit_tangents = tangents / tangent_norms[:, np.newaxis]
    
    # 计算曲率（切线方向变化率）
    curvature = np.diff(unit_tangents, axis=0)
    curvature_norms = np.linalg.norm(curvature, axis=1)
    
    # 平均曲率
    avg_curvature = np.mean(curvature_norms) if len(curvature_norms) > 0 else 0.0
    return avg_curvature



def compute_gae(rewards, values, step_weights, gamma=0.95, lam=0.9):
    """
    rewards:每步即时奖励(10,)
    values:每步状态价值预测(10,)
    step_weights:每步奖励权重(10,)
    返回:GAE加权后的优势值(10,)
    """
    gae = 0
    advantages = np.zeros_like(rewards)
    # 逆序计算（从最后一步到第一步），强化后段权重
    for t in reversed(range(len(rewards))):
        # 后3步提升折扣因子
        gamma_t = 0.98 if t >=7 else gamma
        # 即时优势项
        delta = rewards[t] * step_weights[t] + gamma_t * values[t+1] - values[t] if t < len(rewards)-1 else rewards[t] * step_weights[t] - values[t]
        # GAE累加，融合后续优势
        gae = delta + gamma_t * lam * gae
        advantages[t] = gae
    # 优势值归一化，提升训练稳定性
    advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
    return advantages


def calc_reward(traj, step_info=None, is_terminal=False):
    """改进的奖励函数"""
    rewards = {}
    
    # 1. 目标距离奖励（负向）
    dist_to_goal = np.linalg.norm(traj[-1] - Config.goal_pos)
    r_goal = -dist_to_goal * 0.5
    rewards['goal'] = r_goal
    
    # 2. 碰撞惩罚
    dist_to_obs = np.linalg.norm(traj - Config.obstacle_pos, axis=1)
    min_obs_dist = np.min(dist_to_obs)
    collision = min_obs_dist < Config.obstacle_radius
    
    if collision:
        r_coll = -100.0
        # 额外惩罚靠近障碍物的程度
        r_coll -= (Config.obstacle_radius - min_obs_dist) * 10.0
    else:
        r_coll = 0.0
        # 安全距离奖励（鼓励保持安全距离）
        safety_margin = 0.5
        if min_obs_dist > Config.obstacle_radius + safety_margin:
            r_coll += (min_obs_dist - Config.obstacle_radius - safety_margin) * 0.1
    rewards['collision'] = r_coll
    
    # 3. 轨迹平滑性奖励
    if len(traj) >= 2:
        vel = np.diff(traj, axis=0)
        vel_var = np.var(vel)
        r_smooth = -vel_var * 0.05
        rewards['smooth'] = r_smooth
        
        # 曲率惩罚（鼓励直线）
        curvature = compute_traj_curvature(traj)
        r_curvature = -curvature * 0.02
        rewards['curvature'] = r_curvature
    else:
        rewards['smooth'] = 0.0
        rewards['curvature'] = 0.0
    
    # 4. 渐进奖励（如果提供了上一步信息）
    if step_info is not None:
        prev_dist = step_info.get('prev_dist_to_goal', dist_to_goal)
        progress = prev_dist - dist_to_goal  # 距离减少为正
        r_progress = max(0, progress) * 2.0  # 只奖励正进步
        rewards['progress'] = r_progress
    
    # 5. 最终成功奖励（终端状态）
    if is_terminal:
        success = dist_to_goal < Config.success_threshold
        if success:
            r_success = 50.0
            # 额外奖励：轨迹效率
            traj_length = np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=1))
            optimal_length = np.linalg.norm(Config.goal_pos - traj[0])
            efficiency = optimal_length / max(traj_length, 1e-8)
            r_success += min(efficiency - 1.0, 1.0) * 10.0  # 最多额外10分
            rewards['success'] = r_success
        else:
            rewards['success'] = 0.0
    
    # 6. 探索奖励（鼓励多样性）
    # 计算轨迹的独特性（与其他轨迹的差异）
    if hasattr(calc_reward, 'recent_trajs'):
        if len(calc_reward.recent_trajs) > 0:
            uniqueness = 0.0
            for other_traj in calc_reward.recent_trajs:
                # 简单的L2距离作为差异度量
                diff = np.mean(np.linalg.norm(traj - other_traj, axis=1))
                uniqueness += diff
            
            uniqueness /= len(calc_reward.recent_trajs)
            r_diversity = uniqueness * 0.01  # 小权重鼓励多样性
            rewards['diversity'] = r_diversity
            
    # 步骤4：计算GAE优势值（仅当非终端步时）
    # advantages = np.zeros(10)
    # if not is_terminal:
    #     # 收集全episode的奖励和价值预测
    #     all_rewards = step_info.get('all_rewards', [])
    #     all_rewards.append(total_r)
    #     all_values = step_info.get('all_values', [])
    #     all_values.append(values)
    #     step_info['all_rewards'] = all_rewards
    #     step_info['all_values'] = all_values
    # else:
    #     # 终端步（第10步），计算完整GAE
    #     all_rewards = step_info['all_rewards']
    #     all_values = step_info['all_values']
    #     # 补全价值预测（终端状态价值为0）
    #     all_values.append(0.0)
    #     # 计算每步权重
    #     step_weights = [get_step_weight(i+1) for i in range(10)]
    #     # 计算GAE优势值
    #     advantages = compute_gae(all_rewards, all_values, step_weights)
    
    # 更新最近轨迹缓存
    if not hasattr(calc_reward, 'recent_trajs'):
        calc_reward.recent_trajs = deque(maxlen=100)
    calc_reward.recent_trajs.append(traj.copy())
    
    # 总奖励
    total_reward = sum(rewards.values())
    
    return total_reward, rewards, {
        'dist_to_goal': dist_to_goal,
        'min_obs_dist': min_obs_dist,
        'collision': collision,
        'success': is_terminal and dist_to_goal < Config.success_threshold
    }

def sample_noisy_traj(start_pos=None, goal_pos=None, noise_scale=1.0):
    """生成初始加噪轨迹"""
    if start_pos is None:
        start_pos = Config.start_pos
    if goal_pos is None:
        goal_pos = Config.goal_pos
    
    # 基础轨迹：从起点到目标的直线
    base_traj = np.linspace(start_pos, goal_pos, Config.traj_length)
    
    # 添加噪声（噪声强度随步数衰减）
    t = Config.denoise_steps - 1
    sigma = np.sqrt(1 - Config.alphas_cumprod[t])
    
    # 形状匹配的噪声
    noise = np.random.randn(*base_traj.shape) * sigma * noise_scale
    
    # 保证起点和终点基本不变
    noise[0] *= 0.1
    noise[-1] *= 0.1
    
    return base_traj + noise

# ====================== 5. 经验回放缓冲区 ======================
Experience = namedtuple('Experience', 
                       ['state', 'action', 'reward', 'next_state', 'done', 
                        'log_prob', 'step', 'traj_info'])

class ReplayBuffer:
    """SAC经验回放缓冲区"""
    def __init__(self, capacity, device):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
        self.position = 0
        self.device = device
    
    def push(self, experience):
        """添加经验"""
        self.buffer.append(experience)
    
    def sample(self, batch_size):
        """随机采样批次"""
        if len(self.buffer) < batch_size:
            return None
        
        indices = np.random.randint(0, len(self.buffer), batch_size)
        batch = [self.buffer[i] for i in indices]
        
        # 转换为张量并移动到指定设备
        states = torch.FloatTensor(np.stack([e.state for e in batch])).to(self.device)
        actions = torch.FloatTensor(np.stack([e.action for e in batch])).to(self.device)
        rewards = torch.FloatTensor(np.stack([e.reward for e in batch])).to(self.device)
        
        # 处理next_state（可能为None）
        next_states_list = []
        for e in batch:
            if e.next_state is not None:
                next_states_list.append(e.next_state)
            else:
                # 如果next_state是None，使用当前state
                next_states_list.append(e.state)
        next_states = torch.FloatTensor(np.stack(next_states_list)).to(self.device)
        
        dones = torch.FloatTensor(np.stack([e.done for e in batch])).to(self.device)
        log_probs = torch.FloatTensor(np.stack([e.log_prob for e in batch])).to(self.device)
        steps = torch.LongTensor(np.stack([e.step for e in batch])).to(self.device)
        
        # model_prev_list
        # t_prev_list
        
        return states, actions, rewards, next_states, dones, log_probs, steps
    
    def __len__(self):
        return len(self.buffer)

# ====================== 6. SAC 智能体 ======================
class SACAgent:
    """SAC智能体"""
    def __init__(self, state_dim, action_dim, config):
        self.config = config
        self.state_dim = state_dim
        self.action_dim = action_dim
        
        # 设备 - 确保所有网络使用相同设备
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"SAC Agent using device: {self.device}")
        
        # 网络 - 都放在同一设备上
        self.policy = GaussianPolicy(state_dim, action_dim, config.hidden_dim).to(self.device)
        self.q1 = QNetwork(state_dim, action_dim, config.hidden_dim).to(self.device)
        self.q2 = QNetwork(state_dim, action_dim, config.hidden_dim).to(self.device)
        
        # 目标网络
        self.q1_target = QNetwork(state_dim, action_dim, config.hidden_dim).to(self.device)
        self.q2_target = QNetwork(state_dim, action_dim, config.hidden_dim).to(self.device)
        self.q1_target.load_state_dict(self.q1.state_dict())
        self.q2_target.load_state_dict(self.q2.state_dict())
        
        # 优化器
        self.policy_optimizer = Adam(self.policy.parameters(), lr=config.learning_rate)
        self.q1_optimizer = Adam(self.q1.parameters(), lr=config.learning_rate)
        self.q2_optimizer = Adam(self.q2.parameters(), lr=config.learning_rate)
        
        # 自动学习温度参数α
        self.learn_alpha = config.learn_alpha
        if self.learn_alpha:
            self.log_alpha = torch.tensor(np.log(config.alpha), requires_grad=True, device=self.device)
            self.alpha_optimizer = Adam([self.log_alpha], lr=config.learning_rate)
            self.target_entropy = config.target_entropy
        else:
            self.alpha = config.alpha
        
        # 经验缓冲区
        self.replay_buffer = ReplayBuffer(config.buffer_size, self.device)
        
        # 训练统计
        self.total_steps = 0
        self.episode_rewards = []
        self.success_rate_history = []
        
    def select_action(self, state, deterministic=False):
        """选择动作"""
        # 确保状态在正确的设备上
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            if deterministic:
                action, _, _ = self.policy(state_tensor, deterministic=True)
            else:
                # 这里需要确保forward返回三个值
                result = self.policy(state_tensor, deterministic=False, with_logprob=True)
                if isinstance(result, tuple) and len(result) >= 2:
                    action, log_prob, _ = result[:3]
                else:
                    # 如果forward只返回action，创建一个假的log_prob
                    action = result
                    log_prob = torch.zeros(1, 1).to(self.device)
        
        action_np = action.cpu().numpy()[0]
        
        if deterministic:
            return action_np
        else:
            return action_np, log_prob.item() if isinstance(log_prob, torch.Tensor) else 0.0
    
    def update(self, batch_size=None):
        """更新网络参数"""
        if batch_size is None:
            batch_size = self.config.batch_size
        
        # 从缓冲区采样
        batch = self.replay_buffer.sample(batch_size)
        if batch is None:
            return {}
        
        states, actions, rewards, next_states, dones, old_log_probs, steps = batch
        
        # 更新Q函数->当前价值网络和目标网络差距
        with torch.no_grad():
            # 下一状态的动作和对数概率
            next_actions, next_log_probs, _ = self.policy(next_states)
            
            # 目标Q值
            q1_target = self.q1_target(next_states, next_actions)
            q2_target = self.q2_target(next_states, next_actions)
            q_target = torch.min(q1_target, q2_target)
            
            # 考虑熵的目标值
            alpha = self.log_alpha.exp() if self.learn_alpha else self.alpha
            target_q = rewards + self.config.gamma * (1 - dones) * (q_target - alpha * next_log_probs) #V的定义, q+熵
            
            # sac_base_target = rewards + self.config.gamma * (1 - dones) * (q_target - self.config.alpha * next_log_probs)
    
            # # 步骤2：融合Advantage与时序权重（核心改进）
            # # 1. Advantage归一化（避免尺度干扰）
            # adv_norm = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            # # 2. 叠加Advantage修正项，后段步数通过step_weights放大优势值影响
            # target_q = sac_base_target + adv_norm * step_weights
        
        # 当前Q值
        q1 = self.q1(states, actions)
        q2 = self.q2(states, actions)
        
        # Q函数损失
        q1_loss = F.mse_loss(q1, target_q)
        q2_loss = F.mse_loss(q2, target_q)
        
        # 更新Q网络
        self.q1_optimizer.zero_grad()
        q1_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q1.parameters(), 1.0)
        self.q1_optimizer.step()
        
        self.q2_optimizer.zero_grad()
        q2_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q2.parameters(), 1.0)
        self.q2_optimizer.step()
        
        # 更新策略网络
        new_actions, log_probs, _ = self.policy(states)
        q1_new = self.q1(states, new_actions)
        q2_new = self.q2(states, new_actions)
        q_new = torch.min(q1_new, q2_new)
        
        policy_loss = (alpha * log_probs - q_new).mean()    #最大化V
        
        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.policy_optimizer.step()
        
        # 更新温度参数α log_probs被阻断梯度
        # 既要最大化累积奖励，也要最大化策略熵（保证探索）。α 是 “熵的权重”
        # 若 α 过大：SAC 输出的动作（梯度方向）随机性强，轨迹生成波动大；
        # 若 α 过小：SAC 探索不足，轨迹易陷入局部最优（如绕不开障碍）；

        if self.learn_alpha:
            alpha_loss = -(self.log_alpha * (log_probs.detach() + self.target_entropy)).mean()
            
            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()
        
        # 软更新目标网络
        self.soft_update(self.q1_target, self.q1, self.config.tau)
        self.soft_update(self.q2_target, self.q2, self.config.tau)
        
        # 返回损失信息
        losses = {
            'q1_loss': q1_loss.item(),
            'q2_loss': q2_loss.item(),
            'policy_loss': policy_loss.item(),
            'alpha': alpha.item() if self.learn_alpha else alpha
        }
        
        if self.learn_alpha:
            losses['alpha_loss'] = alpha_loss.item()
        
        return losses
    
    def soft_update(self, target, source, tau):
        """软更新目标网络"""
        for target_param, param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)
    
    def save(self, path):
        """保存模型"""
        checkpoint = {
            'policy': self.policy.state_dict(),
            'q1': self.q1.state_dict(),
            'q2': self.q2.state_dict(),
            'q1_target': self.q1_target.state_dict(),
            'q2_target': self.q2_target.state_dict(),
            'policy_optimizer': self.policy_optimizer.state_dict(),
            'q1_optimizer': self.q1_optimizer.state_dict(),
            'q2_optimizer': self.q2_optimizer.state_dict(),
            'total_steps': self.total_steps,
            'episode_rewards': self.episode_rewards,
            'success_rate_history': self.success_rate_history,
        }
        
        if self.learn_alpha:
            checkpoint['log_alpha'] = self.log_alpha
            checkpoint['alpha_optimizer'] = self.alpha_optimizer.state_dict()
        
        torch.save(checkpoint, path)
    
    def load(self, path):
        """加载模型"""
        checkpoint = torch.load(path, map_location=self.device)
        self.policy.load_state_dict(checkpoint['policy'])
        self.q1.load_state_dict(checkpoint['q1'])
        self.q2.load_state_dict(checkpoint['q2'])
        self.q1_target.load_state_dict(checkpoint['q1_target'])
        self.q2_target.load_state_dict(checkpoint['q2_target'])
        self.policy_optimizer.load_state_dict(checkpoint['policy_optimizer'])
        self.q1_optimizer.load_state_dict(checkpoint['q1_optimizer'])
        self.q2_optimizer.load_state_dict(checkpoint['q2_optimizer'])
        self.total_steps = checkpoint['total_steps']
        self.episode_rewards = checkpoint['episode_rewards']
        self.success_rate_history = checkpoint['success_rate_history']
        
        if self.learn_alpha and 'log_alpha' in checkpoint:
            self.log_alpha = checkpoint['log_alpha'].to(self.device)
            self.alpha_optimizer.load_state_dict(checkpoint['alpha_optimizer'])

# ====================== 7. 训练环境 ======================
class DiffusionGuidingEnv:
    """扩散模型引导环境"""
    def __init__(self, diffusion_model, config, device):
        self.diffusion_model = diffusion_model
        self.config = config
        self.device = device
        self.current_step = None
        self.current_traj = None
        self.goal_pos = config.goal_pos
        self.obstacle_pos = config.obstacle_pos
        self.obstacle_radius = config.obstacle_radius
        
    def reset(self, start_pos=None, goal_pos=None):
        """重置环境"""
        self.current_step = self.config.denoise_steps
        self.current_traj = sample_noisy_traj(start_pos, goal_pos)
        self.goal_pos = goal_pos if goal_pos is not None else self.config.goal_pos
        
        # 初始状态
        state = self._get_state()
        return state, self.current_traj
    
    def step(self, action, deterministic=False):    #输入
        """执行一步"""
        if self.current_step <= 0:
            raise ValueError("Episode already finished")
        
        # 获取当前噪声预测 模型中输入是x_t, t
        noise_pred = self.diffusion_model.predict_noise(
            self.current_traj, self.current_step, device=self.device
        )
        
        # 解码动作生成条件梯度
        cond_grad = decode_action_to_cond_grad(action, self.current_step)
        
        # 引导去噪 model_fn
        sigma_t = get_sigma(self.current_step)
        noise_guided = noise_pred - self.config.guidance_scale * sigma_t * cond_grad
        
        # 执行去噪步骤,获取下一个状态  xt-1->dmp_solver() model_prev_list t_prev_list steps
        # x = self.multistep_dpm_solver_update(x, model_prev_list, t_prev_list, t, step, solver_type=solver_type)
        add_noise = not deterministic  # 训练时加噪声，评估时不加
        self.current_traj = self.diffusion_model.denoise_step(
            self.current_traj, noise_guided, self.current_step, add_noise=add_noise
        )
        
        # 更新步数
        self.current_step -= 1
        
        # 获取下一个状态 
        next_state = self._get_state() if self.current_step > 0 else None
        
        # 判断是否结束
        done = self.current_step == 0
        
        return next_state, self.current_traj, done
        #return next_state, steps, model_prev_list, t_prev_list
    
    def _get_state(self):
        """构建状态向量"""
        if self.current_step <= 0:
            return None
        
        sigma_t = get_sigma(self.current_step)
        noise_pred = self.diffusion_model.predict_noise(
            self.current_traj, self.current_step, device=self.device
        )
        
        # 计算当前性能指标
        reward, _, metrics = calc_reward(self.current_traj, is_terminal=False)
        success_flag = 1.0 if metrics.get('success', False) else 0.0
        
        # 状态组成：[step, sigma_t, traj_flat, noise_pred_flat, goal_pos, success_flag]
        state = np.concatenate([
            [self.current_step / self.config.denoise_steps],  # 归一化的步数
            [sigma_t],
            self.current_traj.flatten(),
            noise_pred.flatten(),
            self.goal_pos,
            [success_flag]
        ])
        
        return state
    
    # 修复后的状态设计：
    # def get_state(x_t, step, diffusion_model, config):
    #     """构建包含完整环境信息的状态"""
        
    #     sigma_t = get_sigma(step)
    #     noise_pred = diffusion_model.predict_noise(x_t, step)
        
    #     # 1. 扩散过程信息（保留）
    #     diffusion_info = np.array([
    #         step / config.denoise_steps,  # 归一化的步数
    #         sigma_t                        # 噪声强度
    #     ])
        
    #     # 2. 当前轨迹状态（保留）
    #     trajectory_info = x_t.flatten()           # 当前轨迹
    #     noise_info = noise_pred.flatten()         # 预测噪声
        
    #     # 3. 核心环境信息（新增！）
    #     goal_info = config.goal_pos               # 目标位置
        
    #     # 障碍物信息（关键修复！）
    #     obstacle_info = np.array([])
    #     for obs in config.obstacles:  # 假设config.obstacles是列表
    #         # 每个障碍物：位置(3) + 半径(1) + 类型(1)
    #         obs_data = np.concatenate([
    #             obs['position'], 
    #             [obs['radius']],
    #             [obs.get('type', 0)]  # 0:球形，1:圆柱形等
    #         ])
    #         obstacle_info = np.concatenate([obstacle_info, obs_data])
        
    #     # 起点信息（重要）
    #     start_info = config.start_pos
        
    #     # 4. 相对位置信息（计算得到的特征）
    #     relative_info = np.array([
    #         np.linalg.norm(x_t[-1] - config.goal_pos),  # 终点到目标距离
    #         np.min(np.linalg.norm(x_t - config.obstacle_pos, axis=1)),  # 最近障碍距离
    #         np.linalg.norm(x_t[0] - config.start_pos)   # 起点偏移
    #     ])
        
    #     # 组合所有状态信息
    #     state = np.concatenate([
    #         diffusion_info,
    #         trajectory_info,
    #         noise_info,
    #         goal_info,
    #         obstacle_info,
    #         start_info,
    #         relative_info
    #     ])
        
    #     return state
    
    def render(self):
        """可视化当前轨迹"""
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # 绘制轨迹
        ax.plot(self.current_traj[:, 0], self.current_traj[:, 1], self.current_traj[:, 2], 
                'b-', linewidth=2, label='Current Trajectory')
        
        # 绘制目标点
        ax.scatter(self.goal_pos[0], self.goal_pos[1], self.goal_pos[2], 
                  color='green', s=200, marker='*', label='Goal')
        
        # 绘制障碍物（简化为球体）
        u = np.linspace(0, 2 * np.pi, 20)
        v = np.linspace(0, np.pi, 20)
        x = self.obstacle_radius * np.outer(np.cos(u), np.sin(v)) + self.obstacle_pos[0]
        y = self.obstacle_radius * np.outer(np.sin(u), np.sin(v)) + self.obstacle_pos[1]
        z = self.obstacle_radius * np.outer(np.ones(np.size(u)), np.cos(v)) + self.obstacle_pos[2]
        ax.plot_surface(x, y, z, color='r', alpha=0.3, label='Obstacle')
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(f'Diffusion Step: {self.current_step}')
        ax.legend()
        ax.grid(True)
        
        plt.show()

# ====================== 8. 训练辅助函数 ======================
def pre_train_diffusion_model(diffusion_model, config, device, epochs=100):
    """预训练扩散模型"""
    print("\n========== 预训练Diffusion模型 ==========")
    
    optimizer_diff = Adam(diffusion_model.parameters(), lr=1e-3)
    
    for epoch in range(epochs):
        batch_size = 32
        x_0 = np.random.randn(batch_size, config.traj_length * config.traj_dim)
        t = np.random.randint(1, config.denoise_steps + 1, size=batch_size)
        noise = np.random.randn(*x_0.shape)
        
        alpha_cumprod = config.alphas_cumprod[t-1].reshape(-1, 1)
        x_t = np.sqrt(alpha_cumprod) * x_0 + np.sqrt(1 - alpha_cumprod) * noise
        
        # 转换为张量并确保在正确设备上
        x_t_tensor = torch.FloatTensor(x_t).to(device)
        t_tensor = torch.LongTensor(t).to(device)
        noise_tensor = torch.FloatTensor(noise).to(device)
        
        # 训练
        optimizer_diff.zero_grad()
        noise_pred = diffusion_model(x_t_tensor, t_tensor)
        loss = F.mse_loss(noise_pred, noise_tensor)
        loss.backward()
        optimizer_diff.step()
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} | Diffusion Loss: {loss.item():.6f}")
    
    print("Diffusion模型预训练完成!")
    return diffusion_model

def visualize_training_progress(agent, episode, env, config):
    """训练过程中的可视化"""
    print(f"\n========== Episode {episode} 测试生成轨迹 ==========")
    
    # 重置环境
    state, traj = env.reset()
    
    # 用确定性策略生成轨迹
    trajectory_history = [traj.copy()]
    done = False
    
    while not done:
        action = agent.select_action(state, deterministic=True)
        next_state, next_traj, done = env.step(action, deterministic=True)
        
        if next_state is not None:
            state = next_state
            traj = next_traj
            trajectory_history.append(traj.copy())
    
    # 计算最终奖励
    final_reward, reward_components, metrics = calc_reward(traj, is_terminal=True)
    
    print(f"最终轨迹奖励: {final_reward:.2f}")
    print(f"是否成功: {metrics.get('success', False)}")
    print(f"终点距离: {metrics.get('dist_to_goal', 0):.2f}")
    print(f"最小障碍距离: {metrics.get('min_obs_dist', 0):.2f}")
    
    # 可视化轨迹
    fig = plt.figure(figsize=(15, 5))
    
    # 子图1: 3D轨迹
    ax1 = fig.add_subplot(131, projection='3d')
    
    # 绘制历史轨迹（透明度渐变）
    for i, hist_traj in enumerate(trajectory_history):
        alpha = (i + 1) / len(trajectory_history)
        ax1.plot(hist_traj[:, 0], hist_traj[:, 1], hist_traj[:, 2], 
                alpha=alpha*0.5, linewidth=1)
    
    # 最终轨迹
    ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], 
            'b-', linewidth=3, label='Final Trajectory')
    
    # 目标点
    ax1.scatter(env.goal_pos[0], env.goal_pos[1], env.goal_pos[2], 
               color='green', s=200, marker='*', label='Goal')
    
    # 障碍物
    u = np.linspace(0, 2 * np.pi, 20)
    v = np.linspace(0, np.pi, 20)
    x = config.obstacle_radius * np.outer(np.cos(u), np.sin(v)) + config.obstacle_pos[0]
    y = config.obstacle_radius * np.outer(np.sin(u), np.sin(v)) + config.obstacle_pos[1]
    z = config.obstacle_radius * np.outer(np.ones(np.size(u)), np.cos(v)) + config.obstacle_pos[2]
    ax1.plot_surface(x, y, z, color='r', alpha=0.3, label='Obstacle')
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title(f'Generated Trajectory (Episode {episode})')
    ax1.legend()
    ax1.grid(True)
    
    # 子图2: 奖励曲线
    ax2 = fig.add_subplot(132)
    if len(agent.episode_rewards) > 10:
        window = min(100, len(agent.episode_rewards))
        moving_avg = np.convolve(agent.episode_rewards, np.ones(window)/window, mode='valid')
        ax2.plot(moving_avg, 'b-', linewidth=2)
        ax2.set_xlabel('Episode')
        ax2.set_ylabel('Moving Average Reward')
        ax2.set_title('Training Reward History')
        ax2.grid(True)
    
    # 子图3: 成功率曲线
    ax3 = fig.add_subplot(133)
    if len(agent.success_rate_history) > 10:
        ax3.plot(agent.success_rate_history, 'g-', linewidth=2)
        ax3.set_xlabel('Episode')
        ax3.set_ylabel('Success Rate')
        ax3.set_title('Success Rate History')
        ax3.grid(True)
        ax3.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()

def visualize_final_results(agent, loss_history):
    """最终训练结果可视化"""
    fig = plt.figure(figsize=(15, 10))
    
    # 1. 奖励曲线
    ax1 = plt.subplot(2, 3, 1)
    if len(agent.episode_rewards) > 10:
        window = min(100, len(agent.episode_rewards))
        moving_avg = np.convolve(agent.episode_rewards, np.ones(window)/window, mode='valid')
        ax1.plot(moving_avg, 'b-', linewidth=2)
        ax1.set_xlabel('Episode')
        ax1.set_ylabel('Moving Average Reward')
        ax1.set_title('Reward History')
        ax1.grid(True)
    
    # 2. 成功率曲线
    ax2 = plt.subplot(2, 3, 2)
    if len(agent.success_rate_history) > 10:
        ax2.plot(agent.success_rate_history, 'g-', linewidth=2)
        ax2.set_xlabel('Episode')
        ax2.set_ylabel('Success Rate')
        ax2.set_title(f'Final Success Rate: {agent.success_rate_history[-1]:.3f}')
        ax2.grid(True)
        ax2.set_ylim([0, 1])
    
    # 3. Q1损失曲线
    ax3 = plt.subplot(2, 3, 3)
    if loss_history['q1']:
        window = min(100, len(loss_history['q1']))
        moving_avg = np.convolve(loss_history['q1'], np.ones(window)/window, mode='valid')
        ax3.plot(moving_avg, 'r-', linewidth=2, label='Q1 Loss')
        ax3.set_xlabel('Update Step')
        ax3.set_ylabel('Loss')
        ax3.set_title('Q1 Loss History')
        ax3.grid(True)
    
    # 4. Q2损失曲线
    ax4 = plt.subplot(2, 3, 4)
    if loss_history['q2']:
        window = min(100, len(loss_history['q2']))
        moving_avg = np.convolve(loss_history['q2'], np.ones(window)/window, mode='valid')
        ax4.plot(moving_avg, 'orange', linewidth=2, label='Q2 Loss')
        ax4.set_xlabel('Update Step')
        ax4.set_ylabel('Loss')
        ax4.set_title('Q2 Loss History')
        ax4.grid(True)
    
    # 5. 策略损失曲线
    ax5 = plt.subplot(2, 3, 5)
    if loss_history['policy']:
        window = min(100, len(loss_history['policy']))
        moving_avg = np.convolve(loss_history['policy'], np.ones(window)/window, mode='valid')
        ax5.plot(moving_avg, 'purple', linewidth=2, label='Policy Loss')
        ax5.set_xlabel('Update Step')
        ax5.set_ylabel('Loss')
        ax5.set_title('Policy Loss History')
        ax5.grid(True)
    
    # 6. 温度参数α曲线
    ax6 = plt.subplot(2, 3, 6)
    if loss_history['alpha']:
        window = min(100, len(loss_history['alpha']))
        moving_avg = np.convolve(loss_history['alpha'], np.ones(window)/window, mode='valid')
        ax6.plot(moving_avg, 'brown', linewidth=2, label='Alpha')
        ax6.set_xlabel('Update Step')
        ax6.set_ylabel('Alpha Value')
        ax6.set_title('Temperature Parameter History')
        ax6.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # 打印最终统计
    print("\n========== 最终训练统计 ==========")
    print(f"总训练步数: {agent.total_steps}")
    print(f"总episode数: {len(agent.episode_rewards)}")
    if len(agent.episode_rewards) >= 100:
        print(f"最终平均奖励: {np.mean(agent.episode_rewards[-100:]):.2f}")
    else:
        print(f"最终平均奖励: {np.mean(agent.episode_rewards):.2f}")
    print(f"最终成功率: {agent.success_rate_history[-1]:.3f}")
    print(f"经验缓冲区大小: {len(agent.replay_buffer)}")

# ====================== 9. 主训练流程 ======================
def train_sac():
    """SAC训练主函数"""
    config = Config()
    
    # 初始化设备 - 使用统一的设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"========== 运行环境 ==========")
    print(f"使用设备: {device}")
    print(f"轨迹长度: {config.traj_length}")
    print(f"去噪步数: {config.denoise_steps}")
    
    # 1. 预训练扩散模型
    diffusion_model = DiffusionModel(
        traj_length=config.traj_length,
        traj_dim=config.traj_dim,
        denoise_steps=config.denoise_steps
    ).to(device)
    
    diffusion_model = pre_train_diffusion_model(diffusion_model, config, device, epochs=100)
    
    # 2. 初始化SAC智能体和环境
    print("\n========== 初始化SAC智能体 ==========")
    agent = SACAgent(config.state_dim, config.action_dim, config)
    env = DiffusionGuidingEnv(diffusion_model, config, device)
    
    # 训练统计
    episode = 0
    best_success_rate = 0.0
    success_window = deque(maxlen=100)  # 最近100个episode的成功率
    reward_history = []
    loss_history = {'q1': [], 'q2': [], 'policy': [], 'alpha': []}
    
    print("\n========== 开始SAC训练 ==========")
    print(f"缓冲区大小: {config.buffer_size}")
    print(f"批量大小: {config.batch_size}")
    print(f"最大步数: {config.max_steps}")
    print(f"预热步数: {config.warmup_steps}")
    
    # 3. 训练循环
    while agent.total_steps < config.max_steps:
        episode += 1
        episode_reward = 0
        episode_success = False
        
        # 重置环境
        if episode % 50 == 0 and episode > config.warmup_steps // config.denoise_steps:
            # 偶尔改变目标位置以增加多样性
            angle = np.random.uniform(0, 2*np.pi)
            distance = np.random.uniform(8, 12)
            random_goal = np.array([
                distance * np.cos(angle),
                distance * np.sin(angle),
                np.random.uniform(-2, 2)
            ])
            state, traj = env.reset(goal_pos=random_goal)
        else:
            state, traj = env.reset()
        
        done = False
        step_info = {'prev_dist_to_goal': float('inf')}
        
        # 轨迹收集循环
        while not done:
            # 探索阶段：随机动作
            if agent.total_steps < config.warmup_steps:
                action = np.random.uniform(-1, 1, config.action_dim)
                # 约束动作
                action[:3] = action[:3] / (np.linalg.norm(action[:3]) + 1e-8)
                action[3:] = 0.5  # 中间权重
                log_prob = 0.0  # 探索阶段不计算log_prob
                '''
                第一步:对原始数据进行完整去噪
                生成随机动作, 缓存noisy_traj,model_prev_list,t_prev_list,steps
                第二步:采样以上全部过程继续生成数据
                '''
            else:
                # SAC策略选择动作
                action_result = agent.select_action(state, deterministic=False)
                if isinstance(action_result, tuple):
                    action, log_prob = action_result
                else:
                    action = action_result
                    log_prob = 0.0
            
            # 执行动作
            next_state, next_traj, done = env.step(action)
            # next_state, next_traj, done = env.step(noisy_traj, model_prev_list, t_prev_list, steps ,action)
            
            # 计算奖励
            if done:
                # 最终状态奖励
                reward, reward_components, metrics = calc_reward(
                    next_traj, step_info, is_terminal=True
                )
                episode_success = metrics.get('success', False)
            else:
                # 中间状态奖励（稀疏奖励设置）
                reward = 0.0  # 稀疏奖励
            
            # 存储经验
            exp = Experience(
                state=state,
                action=action,
                reward=reward,
                next_state=next_state,
                done=done,
                log_prob=log_prob,
                step=env.current_step,
                traj_info=step_info.copy()
            )
            # state_model_prev_list
            # state_t_prev_list
            agent.replay_buffer.push(exp)
            
            # 更新状态 steps==10
            if next_state is not None:
                state = next_state
                traj = next_traj
                # model_prev_list = next_model_prev_list
                # t_prev_list = next_t_prev_list
                # steps = next_steps
                # 更新步信息
                step_info['prev_dist_to_goal'] = np.linalg.norm(traj[-1] - env.goal_pos)
            
            episode_reward += reward
            agent.total_steps += 1
            
            # 定期更新网络（仅在预热后）
            if (agent.total_steps >= config.warmup_steps and 
                agent.total_steps % config.update_every == 0):
                
                # 多次更新以提高数据利用率
                for _ in range(config.num_updates):
                    losses = agent.update()
                    
                    # 记录损失
                    if losses:
                        loss_history['q1'].append(losses.get('q1_loss', 0))
                        loss_history['q2'].append(losses.get('q2_loss', 0))
                        loss_history['policy'].append(losses.get('policy_loss', 0))
                        loss_history['alpha'].append(losses.get('alpha', config.alpha))
        
        # 记录episode结果
        agent.episode_rewards.append(episode_reward)
        success_window.append(1 if episode_success else 0)
        success_rate = np.mean(success_window) if success_window else 0.0
        agent.success_rate_history.append(success_rate)
        
        # 定期输出训练信息
        if episode % 10 == 0:
            avg_reward = np.mean(agent.episode_rewards[-10:]) if len(agent.episode_rewards) >= 10 else episode_reward
            avg_q1_loss = np.mean(loss_history['q1'][-100:]) if loss_history['q1'] else 0
            avg_q2_loss = np.mean(loss_history['q2'][-100:]) if loss_history['q2'] else 0
            
            print(f"Episode {episode:4d} | Steps: {agent.total_steps:6d} | "
                  f"Reward: {episode_reward:7.2f} | Avg Reward: {avg_reward:7.2f} | "
                  f"Success Rate: {success_rate:.3f} | "
                  f"Buffer: {len(agent.replay_buffer):5d} | "
                  f"Q1 Loss: {avg_q1_loss:.4f}")
        
        # 定期保存模型
        if episode % 100 == 0 and success_rate > best_success_rate:
            best_success_rate = success_rate
            agent.save(f"sac_diffusion_best.pth")
            print(f"模型已保存，成功率: {success_rate:.3f}")
        
        # 定期可视化
        if episode % 200 == 0:
            visualize_training_progress(agent, episode, env, config)
    
    # 4. 保存最终模型
    agent.save("sac_diffusion_final.pth")
    print("\n训练完成！最终模型已保存。")
    
    # 5. 可视化训练结果
    visualize_final_results(agent, loss_history)
    
    return agent, diffusion_model

# ====================== 10. 评估函数 ======================
def evaluate_trained_agent(agent_path="sac_diffusion_final.pth"):
    """评估训练好的智能体"""
    config = Config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 加载模型
    diffusion_model = DiffusionModel(
        traj_length=config.traj_length,
        traj_dim=config.traj_dim,
        denoise_steps=config.denoise_steps
    ).to(device)
    
    agent = SACAgent(config.state_dim, config.action_dim, config)
    agent.load(agent_path)
    
    env = DiffusionGuidingEnv(diffusion_model, config, device)
    
    # 多次评估
    n_eval_episodes = 20
    success_count = 0
    total_rewards = []
    
    print(f"\n========== 评估智能体 ({n_eval_episodes}次运行) ==========")
    
    for i in range(n_eval_episodes):
        # 随机起点和终点
        angle = np.random.uniform(0, 2*np.pi)
        distance = np.random.uniform(8, 12)
        random_goal = np.array([
            distance * np.cos(angle),
            distance * np.sin(angle),
            np.random.uniform(-2, 2)
        ])
        
        state, traj = env.reset(goal_pos=random_goal)
        done = False
        
        while not done:
            action = agent.select_action(state, deterministic=True)
            next_state, next_traj, done = env.step(action, deterministic=True)
            
            if next_state is not None:
                state = next_state
                traj = next_traj
        
        # 计算最终性能
        final_reward, _, metrics = calc_reward(traj, is_terminal=True)
        total_rewards.append(final_reward)
        
        if metrics.get('success', False):
            success_count += 1
        
        if i % 5 == 0:
            print(f"运行 {i+1:2d}: 奖励={final_reward:7.2f}, 成功={metrics.get('success', False)}, "
                  f"距离={metrics.get('dist_to_goal', 0):.2f}")
    
    # 统计结果
    avg_reward = np.mean(total_rewards)
    std_reward = np.std(total_rewards)
    success_rate = success_count / n_eval_episodes
    
    print(f"\n========== 评估结果 ==========")
    print(f"平均奖励: {avg_reward:.2f} ± {std_reward:.2f}")
    print(f"成功率: {success_rate:.3f} ({success_count}/{n_eval_episodes})")
    print(f"最小奖励: {np.min(total_rewards):.2f}")
    print(f"最大奖励: {np.max(total_rewards):.2f}")
    
    # 可视化最佳轨迹
    best_idx = np.argmax(total_rewards)
    print(f"\n展示最佳轨迹 (运行 {best_idx+1}, 奖励={total_rewards[best_idx]:.2f})")
    
    # 重新生成最佳轨迹
    angle = np.random.uniform(0, 2*np.pi)
    distance = np.random.uniform(8, 12)
    random_goal = np.array([
        distance * np.cos(angle),
        distance * np.sin(angle),
        np.random.uniform(-2, 2)
    ])
    
    state, traj = env.reset(goal_pos=random_goal)
    trajectory_history = [traj.copy()]
    done = False
    
    while not done:
        action = agent.select_action(state, deterministic=True)
        next_state, next_traj, done = env.step(action, deterministic=True)
        
        if next_state is not None:
            state = next_state
            traj = next_traj
            trajectory_history.append(traj.copy())
    
    # 可视化
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 历史轨迹
    for i, hist_traj in enumerate(trajectory_history):
        alpha = (i + 1) / len(trajectory_history)
        ax.plot(hist_traj[:, 0], hist_traj[:, 1], hist_traj[:, 2], 
                alpha=alpha*0.5, linewidth=1, color='blue')
    
    # 最终轨迹
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], 
            'b-', linewidth=3, label='Generated Trajectory')
    
    # 目标点
    ax.scatter(random_goal[0], random_goal[1], random_goal[2], 
               color='green', s=200, marker='*', label='Goal')
    
    # 障碍物
    u = np.linspace(0, 2 * np.pi, 20)
    v = np.linspace(0, np.pi, 20)
    x = config.obstacle_radius * np.outer(np.cos(u), np.sin(v)) + config.obstacle_pos[0]
    y = config.obstacle_radius * np.outer(np.sin(u), np.sin(v)) + config.obstacle_pos[1]
    z = config.obstacle_radius * np.outer(np.ones(np.size(u)), np.cos(v)) + config.obstacle_pos[2]
    ax.plot_surface(x, y, z, color='r', alpha=0.3, label='Obstacle')
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Best Generated Trajectory (Reward: {total_rewards[best_idx]:.2f})')
    ax.legend()
    ax.grid(True)
    
    plt.show()
    
    return agent, diffusion_model

# ====================== 11. 主程序入口 ======================
if __name__ == "__main__":
    print("=" * 60)
    print("SAC + Diffusion Model 轨迹生成系统")
    print("=" * 60)
    
    # 训练模式
    train_mode = True  # 设置为False进行仅评估
    
    if train_mode:
        try:
            # 训练新模型
            agent, diffusion_model = train_sac()
            
            # 评估训练好的模型
            evaluate_trained_agent("sac_diffusion_final.pth")
        except Exception as e:
            print(f"训练过程中出现错误: {e}")
            import traceback
            traceback.print_exc()
    else:
        # 仅评估已训练模型
        try:
            evaluate_trained_agent()
        except Exception as e:
            print(f"评估过程中出现错误: {e}")
            import traceback
            traceback.print_exc()
    
    print("\n程序执行完成！")

SAC + Diffusion Model 轨迹生成系统
========== 运行环境 ==========
使用设备: cuda
轨迹长度: 50
去噪步数: 10

========== 预训练Diffusion模型 ==========
Epoch   0 | Diffusion Loss: 1.413577
Epoch  20 | Diffusion Loss: 0.991536
Epoch  40 | Diffusion Loss: 1.021127
Epoch  60 | Diffusion Loss: 0.984917
Epoch  80 | Diffusion Loss: 1.031165
Diffusion模型预训练完成!

========== 初始化SAC智能体 ==========
SAC Agent using device: cuda

========== 开始SAC训练 ==========
缓冲区大小: 100000
批量大小: 256
最大步数: 100000
预热步数: 5000
Episode   10 | Steps:    100 | Reward: -115.00 | Avg Reward: -112.44 | Success Rate: 0.000 | Buffer:   100 | Q1 Loss: 0.0000
Episode   20 | Steps:    200 | Reward: -114.22 | Avg Reward: -112.21 | Success Rate: 0.000 | Buffer:   200 | Q1 Loss: 0.0000
Episode   30 | Steps:    300 | Reward: -112.02 | Avg Reward: -111.99 | Success Rate: 0.000 | Buffer:   300 | Q1 Loss: 0.0000
Episode   40 | Steps:    400 | Reward: -109.61 | Avg Reward: -111.34 | Success Rate: 0.000 | Buffer:   400 | Q1 Loss: 0.0000
Episode   50 | Steps:    500 | Re

KeyboardInterrupt: 